In [41]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
nltk.download('punkt_tab')

import spacy

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/julienrm/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [42]:
df_fr = pd.read_csv('data/small_vocab_fr.txt', sep='\t', names=['text'])
df_fr.head()

,text
0,new jersey est parfois calme pendant l' automn...
1,les états-unis est généralement froid en juill...
2,"california est généralement calme en mars , et..."
3,"les états-unis est parfois légère en juin , et..."
4,"votre moins aimé fruit est le raisin , mais mo..."


In [43]:
df_en = pd.read_csv('data/small_vocab_en.txt', sep='\t', names=['text'])
df_en.head()

,text
0,"new jersey is sometimes quiet during autumn , ..."
1,the united states is usually chilly during jul...
2,"california is usually quiet during march , and..."
3,the united states is sometimes mild during jun...
4,"your least liked fruit is the grape , but my l..."


In [44]:
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text, remove_stopwords=False, language='french', remove_punctuation=False):
    """
    Preprocess text by cleaning and tokenizing
    """
    # Basic text cleaning
    text = text.strip() # .lower()
    def fix_punctuation_spacing(text):
        # Apostrophe: no spaces around it
        if text.find("'") != -1:
            text = text.replace(" '", "'").replace("' ", "'")

        # Hyphen in compound words: no spaces around it
        # Em dash (—) or en dash (–): space before and after for sentence breaks
        if text.find("-") != -1:
            # First handle spaced dashes (likely sentence breaks)
            text = text.replace(" - ", " — ")  # Convert to em dash
            text = text.replace(" -", " —").replace("- ", "— ")
            
            # Replace em dashes back to spaced format
            text = text.replace("—", " — ")
            
            # Clean up multiple spaces around em dashes
            text = re.sub(r'\s*—\s*', ' — ', text)
        
        # Comma: no space before, one space after
        if text.find(",") != -1:
            text = text.replace(" ,", ",")
            # Add space after comma if not already there
            text = re.sub(r',(?!\s)', ', ', text)
            # Fix multiple spaces after comma
            text = text.replace(",  ", ", ")

        # Period: no space before, one space after (except end of text)
        if text.find(".") != -1:
            text = text.replace(" .", ".")
            # Add space after period if not already there and not at end
            text = re.sub(r'\.(?!\s|$)', '. ', text)
            # Fix multiple spaces after period
            text = text.replace(".  ", ". ")
        
        # Semicolon: no space before, one space after
        if text.find(";") != -1:
            text = text.replace(" ;", ";")
            text = re.sub(r';(?!\s)', '; ', text)
            text = text.replace(";  ", "; ")
        
        # Colon: no space before, one space after
        if text.find(":") != -1:
            text = text.replace(" :", ":")
            text = re.sub(r':(?!\s)', ': ', text)
            text = text.replace(":  ", ": ")
        
        # Question mark: no space before, one space after
        if text.find("?") != -1:
            text = text.replace(" ?", "?")
            text = re.sub(r'\?(?!\s|$)', '? ', text)
            text = text.replace("?  ", "? ")
        
        # Exclamation mark: no space before, one space after
        if text.find("!") != -1:
            text = text.replace(" !", "!")
            text = re.sub(r'!(?!\s|$)', '! ', text)
            text = text.replace("!  ", "! ")
        
        # Opening parenthesis: one space before (if not at start), no space after
        if text.find("(") != -1:
            text = re.sub(r'(?<!\s)(?<!^)\(', ' (', text)  # Add space before if not already there
            text = text.replace("( ", "(")  # Remove space after
            text = text.replace("  (", " (")  # Fix double spaces
        
        # Closing parenthesis: no space before, one space after (if not at end)
        if text.find(")") != -1:
            text = text.replace(" )", ")")
            text = re.sub(r'\)(?!\s|$|[.,;:!?])', ') ', text)  # Add space after unless at end or before punctuation
            text = text.replace(")  ", ") ")
        
        # Clean up any multiple spaces
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()

    text = fix_punctuation_spacing(text)

    # Remove digits
    # cleaned_text = ''.join(char for char in text if not char.isdigit())
    
    # Remove punctuation
    # if remove_punctuation:
    #     for punctuation in string.punctuation:
    #         cleaned_text = cleaned_text.replace(punctuation, ' ')

    # Tokenize
    word_tokens = word_tokenize(text, language=language)
    
    # if asked to remove stopwords
    if remove_stopwords:
        print("Removing stopwords")
        # Remove stop words
        if language == 'french':
            stop_words = set(stopwords.words('french'))
        else:
            stop_words = set(stopwords.words('english'))
    
        tokens_cleaned = [w for w in word_tokens if w not in stop_words and len(w) > 0]
    
        return tokens_cleaned
    
    # # Load relevant language model
    # if language == 'french':
        
    #     nlp = spacy.load('fr_core_news_sm')
    # else:
    #     nlp = spacy.load('en_core_web_sm')

    # def process_text(text):
    #     # this is processing part.
    #     doc = nlp(text)

    #     # Filtering step
    #     filtered_tokens = [token.text for token in doc if not token.is_stop]

    #     print("Filtered Tokens:", filtered_tokens)
    word_tokens = [wt for wt in word_tokens if len(wt) > 0]
    # print(word_tokens)

    return word_tokens

# Apply preprocessing to French data
df_fr['tokens'] = df_fr['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='french'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_fr = TfidfVectorizer()
# Training it on the texts
weighted_df_fr = pd.DataFrame(tf_idf_vectorizer_fr.fit_transform(df_fr['text']).toarray(),
                    columns = tf_idf_vectorizer_fr.get_feature_names_out())

# Apply preprocessing to English data
df_en['tokens'] = df_en['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='english'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_en = TfidfVectorizer()

# Training it on the texts
weighted_df_en = pd.DataFrame(tf_idf_vectorizer_en.fit_transform(df_en['text']).toarray(),
                    columns = tf_idf_vectorizer_en.get_feature_names_out())

print("French preprocessing complete")
print("English preprocessing complete")

# Rename columns to be specific to each language
df_fr_renamed = df_fr.rename(columns={'text': 'text_fr', 'tokens': 'tokens_fr'})
df_en_renamed = df_en.rename(columns={'text': 'text_en', 'tokens': 'tokens_en'})

# Combine side by side
df_fr_en = pd.concat([df_fr_renamed, df_en_renamed], axis=1)

import csv
with open('data/cleaned_texts.csv', 'w', newline='') as csvfile:
    df_fr_en.to_csv(csvfile, index=False)

French preprocessing complete
English preprocessing complete


In [45]:
print("\nFrench:", weighted_df_fr.shape[1], "\nEnglish:", weighted_df_en.shape[1])


French: 321 
English: 196


In [46]:
# Combine side by side

df_fr_en

,text_fr,tokens_fr,text_en,tokens_en
0,new jersey est parfois calme pendant l' automn...,"[new, jersey, est, parfois, calme, pendant, l'...","new jersey is sometimes quiet during autumn , ...","[new, jersey, is, sometimes, quiet, during, au..."
1,les états-unis est généralement froid en juill...,"[les, états-unis, est, généralement, froid, en...",the united states is usually chilly during jul...,"[the, united, states, is, usually, chilly, dur..."
2,"california est généralement calme en mars , et...","[california, est, généralement, calme, en, mar...","california is usually quiet during march , and...","[california, is, usually, quiet, during, march..."
3,"les états-unis est parfois légère en juin , et...","[les, états-unis, est, parfois, légère, en, ju...",the united states is sometimes mild during jun...,"[the, united, states, is, sometimes, mild, dur..."
4,"votre moins aimé fruit est le raisin , mais mo...","[votre, moins, aimé, fruit, est, le, raisin, ,...","your least liked fruit is the grape , but my l...","[your, least, liked, fruit, is, the, grape, ,,..."
...,...,...,...,...
137855,"la france est jamais occupée en mars , et il e...","[la, france, est, jamais, occupée, en, mars, ,...","france is never busy during march , and it is ...","[france, is, never, busy, during, march, ,, an..."
137856,"l' inde est parfois belle au printemps , et il...","[l'inde, est, parfois, belle, au, printemps, ,...","india is sometimes beautiful during spring , a...","[india, is, sometimes, beautiful, during, spri..."
137857,"l' inde est jamais mouillé pendant l' été , ma...","[l'inde, est, jamais, mouillé, pendant, l'été,...","india is never wet during summer , but it is s...","[india, is, never, wet, during, summer, ,, but..."
137858,"la france est jamais froid en janvier , mais i...","[la, france, est, jamais, froid, en, janvier, ...","france is never chilly during january , but it...","[france, is, never, chilly, during, january, ,..."


In [47]:
print("Before dropping duplicates:", df_fr_en.shape)
df_fr_en = df_fr_en.drop_duplicates(subset="text_fr", keep="first").reset_index(drop=True)
print("After dropping duplicates:", df_fr_en.shape)

Before dropping duplicates: (137860, 4)
After dropping duplicates: (120806, 4)


In [48]:
df_fr_en.text_fr[137855]

KeyError: 137855

In [ ]:
df_fr_en.tokens_fr[137855]

['la',
 'france',
 'est',
 'jamais',
 'occupée',
 'en',
 'mars',
 ',',
 'et',
 'il',
 'est',
 'parfois',
 'agréable',
 'en',
 'septembre',
 '.']

In [ ]:
weighted_df_en

,am,and,animal,animals,apple,apples,april,are,aren,august,...,when,where,white,why,winter,wonderful,would,yellow,you,your
0,0.0,0.180260,0.0,0.0,0.000000,0.0,0.366932,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.162972,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
2,0.0,0.175169,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
3,0.0,0.175852,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.000000,0.0,0.0,0.304045,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.255303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137855,0.0,0.185612,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137856,0.0,0.191898,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137857,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.377138,0.0,0.0,0.0,0.0,0.000000
137858,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000


In [ ]:
df_fr_en['text_fr'][23]

"paris est doux pendant l' été , mais il est généralement occupé en avril ."

In [ ]:
df_fr_en['text_fr']
df_fr_en['text_en']

0         new jersey is sometimes quiet during autumn , ...
1         the united states is usually chilly during jul...
2         california is usually quiet during march , and...
3         the united states is sometimes mild during jun...
4         your least liked fruit is the grape , but my l...
                                ...                        
137855    france is never busy during march , and it is ...
137856    india is sometimes beautiful during spring , a...
137857    india is never wet during summer , but it is s...
137858    france is never chilly during january , but it...
137859    the orange is her favorite fruit , but the ban...
Name: text_en, Length: 137860, dtype: object

In [ ]:
df_fr_en["tokens_fr"]

0         [new, jersey, est, parfois, calme, pendant, l'...
1         [les, états-unis, est, généralement, froid, en...
2         [california, est, généralement, calme, en, mar...
3         [les, états-unis, est, parfois, légère, en, ju...
4         [votre, moins, aimé, fruit, est, le, raisin, ,...
                                ...                        
137855    [la, france, est, jamais, occupée, en, mars, ,...
137856    [l'inde, est, parfois, belle, au, printemps, ,...
137857    [l'inde, est, jamais, mouillé, pendant, l'été,...
137858    [la, france, est, jamais, froid, en, janvier, ...
137859    [l'orange, est, son, fruit, préféré, ,, mais, ...
Name: tokens_fr, Length: 137860, dtype: object

In [ ]:
from LSTM_translator import train_translator_from_tokens, load_translator_for_inference, test_translation

In [ ]:

import tensorflow as tf

In [ ]:
# Check current setup
print("TensorFlow version:", tf.__version__)
print("Built with MPS:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
Built with MPS: []


In [ ]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


# Training

# LSTM

In [ ]:
# # Your data structure: df with columns ['tokens_fr', 'tokens_en']
# # Example: df.iloc[0]['tokens_fr'] = ['new', 'jersey', 'est', 'parfois', 'calme', ...]

# tf.random.set_seed(42)

# # Train the model
# # with tf.device('/GPU:0'):
# with tf.device('/CPU:0'):
#     translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

# # Save for later use
# translator.save_model('lstm_translator')

# # Test some translations
# test_translation(translator, df_fr_en, n_examples=5)


In [ ]:
translator = load_translator_for_inference('lstm_translator')

# # Method 1: Preprocess then translate tokens manually
# french_tokens = translator.preprocess_french_phrase("Bonjour, comment allez-vous ?")
# english_tokens = translator.translate_tokens(french_tokens)
# print(english_tokens)

# Method 2: Translate entire sentence directly
english_tokens = translator.translate_sentence("il aime la mangue ?")
print(english_tokens)

Loading model from lstm_translator...
Model PreLoaded Successfully!
Translator instance created.
Vocabularies loaded.
Rebuilding model architecture...
Building the model with attention=True, bidirectional=True...
Model built with 29,875,916 parameters
Loading weights from lstm_translator_main.h5...
Weights loaded successfully. Building inference helpers...
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
['she', 'likes', 'the', 'united', 'states', 'is', 'wet', 'during', '.']


In [49]:
# After kernel restart - Fresh test
from LSTM_translator import load_translator_for_inference

translator_loaded = load_translator_for_inference('lstm_translator')
test_tokens = ['new', 'jersey', 'est', 'parfois', 'calme']
result = translator_loaded.translate_tokens(test_tokens)
print(result)

Loading model from lstm_translator...
Model PreLoaded Successfully!
Translator instance created.
Vocabularies loaded.
Rebuilding model architecture...
Building the model with attention=True, bidirectional=True...
Model built with 274,892 parameters
Loading weights from lstm_translator_main.h5...
Weights loaded successfully. Building inference helpers...
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
['new', 'jersey', 'is', 'sometimes', 'quiet', 'snowy', 'snowy', 'snowy', 'winter', 'in', 'winter', '.']


In [50]:
# Test cell - Loading saved model and attempting prediction
import tensorflow as tf
from LSTM_translator import load_translator_for_inference
import numpy as np
# Load the saved model 
translator_loaded = load_translator_for_inference('lstm_translator')
# Test with a simple French sentence
test_tokens = ['new', 'jersey', 'est', 'parfois', 'calme']
print(f"Testing translation of: {test_tokens}")
# Try to translate (this should fail with same error)
try:
    result = translator_loaded.translate_tokens(test_tokens)
    print(f"Translation: {result}")
except AttributeError as e:
    print(f"Expected error: {e}")
    print("Confirming the issue exists with loaded models too")


Loading model from lstm_translator...
Model PreLoaded Successfully!
Translator instance created.
Vocabularies loaded.
Rebuilding model architecture...
Building the model with attention=True, bidirectional=True...
Model built with 274,892 parameters
Loading weights from lstm_translator_main.h5...
Weights loaded successfully. Building inference helpers...
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
Testing translation of: ['new', 'jersey', 'est', 'parfois', 'calme']
Translation: ['new', 'jersey', 'is', 'sometimes', 'quiet', 'snowy', 'snowy', 'snowy', 'winter', 'in', 'winter', '.']


In [51]:

translator = load_translator_for_inference('lstm_translator')

# Translate new French tokens but change to words for vocabulary
french_tokens = ['bonjour', 'comment', 'allez', 'vous']
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['hello', 'how', 'are', 'you']
french_tokens = ["C'est meilleur la banane ?"]
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['Is the banana better?']

Loading model from lstm_translator...
Model PreLoaded Successfully!
Translator instance created.
Vocabularies loaded.
Rebuilding model architecture...
Building the model with attention=True, bidirectional=True...
Model built with 274,892 parameters
Loading weights from lstm_translator_main.h5...
Weights loaded successfully. Building inference helpers...
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
['they', 'like', 'a', 'old', 'red', 'automobile', 'automobile', 'automobile', '?', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', 'october', '?', '?', 'october', '?', 'october', '?', 'october', '?']
['they', 'like', 'strawberries', 'strawberries', 'winter'

In [52]:
# Method 2: Translate entire sentence directly

print(translator.translate_sentence("c'est bien la banane ?"))
print(translator.translate_sentence("parfois calme parfois neigeux"))

print(translator.translate_sentence("new jersey est parfois calme pendant l' automne , et il est neigeux en avril ."))
print(translator.translate_sentence("le pamplemousse est votre fruit le plus aimé , mais le raisin est leur plus aimé ."))

['did', 'she', 'like', 'the', 'big', 'yellow', 'car', 'automobile', '?']
['they', 'like', 'limes', ',', 'strawberries', '.']
['new', 'jersey', 'is', 'sometimes', 'quiet', 'snowy', 'during', 'fall', ',', 'and', 'it', 'is', 'snowy', 'april', '.']
['the', 'grapefruit', 'is', 'your', 'most', 'loved', 'fruit', ',', 'but', 'the', 'grape', 'is', 'their', 'most', 'loved', '.']


In [36]:
# # Recover the exact validation set used during training
# from sklearn.model_selection import train_test_split
# # Recreate the same split (using the same parameters as during training)
# df_temp, df_val = train_test_split(df_fr_en, test_size=1000, random_state=42)
# print(f"Recovered validation set: {len(df_val)} samples")
# # Test your model on the validation set
# def calculate_validation_accuracy(translator, df_val, n_samples=100):
#     """Calculate translation accuracy on validation set"""
#     correct = 0
#     total = min(n_samples, len(df_val))
#     for i in range(total):
#         fr_tokens = df_val.iloc[i]['tokens_fr']
#         true_en_tokens = df_val.iloc[i]['tokens_en']
#         try:
#             predicted_tokens = translator.translate_tokens(fr_tokens)
#             # Simple accuracy: exact match
#             if predicted_tokens == true_en_tokens:
#                 correct += 1
#             if i % 20 == 0:
#                 print(f"Sample {i+1}:")
#                 print(f"  French: {' '.join(fr_tokens)}")
#                 print(f"  True: {' '.join(true_en_tokens)}")
#                 print(f"  Predicted: {' '.join(predicted_tokens)}")
#                 print(f"  Match: {'✓' if predicted_tokens == true_en_tokens else '✗'}")
#         except Exception as e:
#             print(f"Translation failed for sample {i}: {e}")
#     accuracy = correct / total
#     print(f"\nValidation Accuracy: {accuracy:.4f} ({correct}/{total})")
#     return accuracy
# # Use it
# accuracy = calculate_validation_accuracy(translator, df_val, n_samples=1000)


Validation Accuracy: 0.7500 (750/1000)

Sample 81:
  French: la france est belle au mois de novembre , mais il est généralement doux en été .
  True: france is nice during november , but it is usually mild in summer .
  Predicted: france is beautiful during november , but it is usually mild in summer .
...
  Predicted: the strawberry is her most loved fruit , but the grape is my most loved .
  Match: ✗

Validation Accuracy: 0.7500 (750/1000)

In [37]:
import nltk

hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']
#there may be several references
BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

0.4548019047027907


In [38]:
    from sklearn.model_selection import train_test_split
    import nltk
    from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
    from rouge_score import rouge_scorer


    # --- Split exactly as before ---
    df_temp, df_val = train_test_split(df_fr_en, test_size=1000, random_state=42)
    print(f"Recovered validation set: {len(df_val)} samples")

    # --- Metric helpers ---

    # BLEU smoothing is important for short sentences to avoid 0s from missing higher-order n-grams.
    _bleu_smoother = SmoothingFunction().method3

    # ROUGE scorer: we’ll report F1 for each variant (more stable for translation than recall-only)
    _rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    def _compute_bleu_per_sentence(reference_tokens, hypothesis_tokens):
        """
        reference_tokens: list[str]
        hypothesis_tokens: list[str]
        Returns sentence BLEU (0..1)
        """
        return sentence_bleu(
            [reference_tokens],                   # list of references; you can add more if you have them
            hypothesis_tokens,
            smoothing_function=_bleu_smoother
        )

    def _compute_rouge_f1(reference_tokens, hypothesis_tokens):
        """
        Compute ROUGE-1/2/L F1 between lists of tokens.
        Returns dict with keys: rouge1, rouge2, rougeL (F1 each).
        """
        # ROUGE expects strings; join tokens with spaces
        ref = " ".join(reference_tokens)
        hyp = " ".join(hypothesis_tokens)
        s = _rouge.score(ref, hyp)
        return {
            "rouge1": s["rouge1"].fmeasure,
            "rouge2": s["rouge2"].fmeasure,
            "rougeL": s["rougeL"].fmeasure,
        }

    def calculate_validation_metrics(translator, df_val, n_samples=1000, log_every=20):
        """
        Evaluate on the validation set with:
        - Exact-match accuracy
        - Mean sentence BLEU
        - Corpus BLEU
        - Mean ROUGE-1/2/L (F1)

        Returns a dict of aggregate metrics.
        """
        correct = 0
        total = min(n_samples, len(df_val))
        bleu_scores = []
        rouge1_scores = []
        rouge2_scores = []
        rougeL_scores = []

        # For corpus BLEU we need aggregated references and hypotheses
        corpus_references = []  # list of list-of-references (each reference is a list of tokens)
        corpus_hypotheses = []  # list of hypothesis token lists

        for i in range(total):
            fr_tokens = df_val.iloc[i]['tokens_fr']
            true_en_tokens = df_val.iloc[i]['tokens_en']
            try:
                predicted_tokens = translator.translate_tokens(fr_tokens)

                # --- Exact match accuracy ---
                if predicted_tokens == true_en_tokens:
                    correct += 1

                # --- Sentence BLEU ---
                bleu = _compute_bleu_per_sentence(true_en_tokens, predicted_tokens)
                bleu_scores.append(bleu)

                # --- ROUGE (F1) ---
                r = _compute_rouge_f1(true_en_tokens, predicted_tokens)
                rouge1_scores.append(r["rouge1"])
                rouge2_scores.append(r["rouge2"])
                rougeL_scores.append(r["rougeL"])

                # --- Accumulate for corpus BLEU ---
                corpus_references.append([true_en_tokens])  # wrap in list: potentially multiple refs
                corpus_hypotheses.append(predicted_tokens)

                # --- Optional logging ---
                if i % log_every == 0:
                    print(f"Sample {i+1}:")
                    print(f"  French:    {' '.join(fr_tokens)}")
                    print(f"  True:      {' '.join(true_en_tokens)}")
                    print(f"  Predicted: {' '.join(predicted_tokens)}")
                    print(f"  Match:     {'✓' if predicted_tokens == true_en_tokens else '✗'}")
                    print(f"  BLEU:      {bleu:.4f} | ROUGE-1/2/L (F1): "
                        f"{r['rouge1']:.4f}/{r['rouge2']:.4f}/{r['rougeL']:.4f}")

            except Exception as e:
                print(f"Translation failed for sample {i}: {e}")

        # --- Aggregates ---
        accuracy = correct / total if total > 0 else 0.0
        mean_sentence_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
        # corpus BLEU (usually reported for MT), also with smoothing to handle short texts robustly
        corpus_bleu_score = corpus_bleu(corpus_references, corpus_hypotheses, smoothing_function=_bleu_smoother) if corpus_hypotheses else 0.0
        mean_rouge1 = sum(rouge1_scores) / len(rouge1_scores) if rouge1_scores else 0.0
        mean_rouge2 = sum(rouge2_scores) / len(rouge2_scores) if rouge2_scores else 0.0
        mean_rougeL = sum(rougeL_scores) / len(rougeL_scores) if rougeL_scores else 0.0

        # --- Report ---
        print("\n=== Validation Summary ===")
        print(f"Accuracy (exact match): {accuracy:.4f} ({correct}/{total})")
        print(f"Mean sentence BLEU:     {mean_sentence_bleu:.4f}")
        print(f"Corpus BLEU:            {corpus_bleu_score:.4f}")
        print(f"ROUGE-1/2/L (F1 mean):  {mean_rouge1:.4f} / {mean_rouge2:.4f} / {mean_rougeL:.4f}")

        return {
            "accuracy": accuracy,
            "mean_sentence_bleu": mean_sentence_bleu,
            "corpus_bleu": corpus_bleu_score,
            # "mean_rouge1_f1": mean_rouge1,
            # "mean_rouge2_f1": mean_rouge2,
            "mean_rougeL_f1": mean_rougeL,
        }

    # --- Example usage (replaces your previous call) ---
    metrics = calculate_validation_metrics(translator, df_val, n_samples=1000)

Recovered validation set: 1000 samples
Sample 1:
  French:    chine est généralement occupé en septembre , mais il est parfois froid au printemps .
  True:      china is usually busy during september , but it is sometimes cold in spring .
  Predicted: china is usually busy during september , but it is sometimes chilly in spring .
  Match:     ✗
  BLEU:      0.8003 | ROUGE-1/2/L (F1): 0.9231/0.8333/0.9231
Sample 21:
  French:    france est merveilleux au mois de novembre , et il est neigeux à l'automne .
  True:      france is wonderful during november , and it is snowy in fall .
  Predicted: france is pleasant during november , and it is your in fall .
  Match:     ✗
  BLEU:      0.5266 | ROUGE-1/2/L (F1): 0.8182/0.6000/0.8182
Sample 41:
  French:    la france est généralement chaud en février , et il est sec en juin .
  True:      france is usually hot during february , and it is dry in june .
  Predicted: france is usually wet during february , and it is usually chilly in june .
  Ma

KeyboardInterrupt: 

## GRU

In [ ]:
from GRU_translator import GRUTranslator, train_translator_from_tokens

# Train GRU model (uses same config as LSTM)
gru_translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

# Save and load
gru_translator.save_model('gru_translator')
loaded_gru = GRUTranslator.load_model('gru_translator')

# Translate
result = gru_translator.translate_tokens(['hello', 'world'])

=== Training Configuration ===
Embedding Dim: 512
Hidden Units: 1024
Max Vocab Size: 50000
Attention: True
Bidirectional: True
Teacher Forcing: True
Scheduled Sampling: True
Epochs: 18, Patience: 10
Processing tokenized data...
Dataset size after filtering: 137860
Training set: 109488
Test set: 27372
Validation set: 1000
Building vocabularies...
French vocabulary size: 361
English vocabulary size: 204
Converting tokens to sequences...
Building the model with attention=True, bidirectional=True...
Model built with 23,066,316 parameters
Starting training...
Epoch 1/18
214/214 ━━━━━━━━━━━━━━━━━━━━ 144729s 679s/step - accuracy: 0.9071 - loss: 0.3626 - val_accuracy: 0.9670 - val_loss: 0.1003 - learning_rate: 0.0010
Epoch 2/18
 29/214 ━━━━━━━━━━━━━━━━━━━━ 198:47:03 3868s/step - accuracy: 0.9669 - loss: 0.1032

In [24]:
df_fr_en['text_fr'][14230]

'le pamplemousse est votre fruit le plus aimé , mais le raisin est leur plus aimé .'

In [25]:
df_fr_en["text_en"]

0         new jersey is sometimes quiet during autumn , ...
1         the united states is usually chilly during jul...
2         california is usually quiet during march , and...
3         the united states is sometimes mild during jun...
4         your least liked fruit is the grape , but my l...
                                ...                        
137855    france is never busy during march , and it is ...
137856    india is sometimes beautiful during spring , a...
137857    india is never wet during summer , but it is s...
137858    france is never chilly during january , but it...
137859    the orange is her favorite fruit , but the ban...
Name: text_en, Length: 137860, dtype: object

In [20]:
import nltk

hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']
#there may be several references
BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

0.4548019047027907


In [26]:
import pandas as pd
from collections import Counter

# Assuming your data is in df_fr_en["text_en"]
all_text = " ".join(df_fr_en["text_en"].astype(str))
words = all_text.split(" ")
word_counts = Counter(words)

# Display results
for word, count in word_counts.most_common():
    print(f"{word}: {count}")

is: 205858
,: 140897
.: 129039
in: 75525
it: 75137
during: 74933
the: 67628
but: 63987
and: 59850
sometimes: 37746
usually: 37507
never: 37500
least: 27564
favorite: 27371
fruit: 27105
most: 14934
loved: 13666
liked: 13546
new: 12197
paris: 11334
india: 11277
united: 11270
states: 11270
california: 11250
jersey: 11225
france: 11170
china: 10953
he: 10786
she: 10786
grapefruit: 10118
your: 9734
my: 9700
his: 9700
her: 9700
fall: 9134
june: 9133
spring: 9102
january: 9090
winter: 9038
march: 9023
autumn: 9004
may: 8995
nice: 8984
september: 8958
july: 8956
april: 8954
november: 8951
summer: 8948
december: 8945
february: 8942
our: 8932
their: 8932
freezing: 8928
pleasant: 8916
beautiful: 8915
october: 8910
snowy: 8898
warm: 8890
cold: 8878
wonderful: 8808
dry: 8794
busy: 8791
august: 8789
chilly: 8770
rainy: 8761
mild: 8743
wet: 8726
relaxing: 8696
quiet: 8693
hot: 8639
dislikes: 7314
likes: 7314
limes: 5554
mangoes: 5549
lemons: 5533
grapes: 5525
apples: 5452
oranges: 5452
strawberries: 

In [27]:
import pandas as pd
from collections import Counter

# Assuming your data is in df_fr_en["text_en"]
all_text = " ".join(df_fr_en["text_fr"].astype(str))
words = all_text.split(" ")
word_counts = Counter(words)

# Display results
for word, count in word_counts.most_common():
    print(f"{word}: {count}")

est: 196809
.: 135619
,: 123135
en: 105768
il: 84079
les: 65255
mais: 63987
et: 59851
la: 49861
parfois: 37746
jamais: 37215
le: 35306
l': 32917
généralement: 31292
moins: 27557
au: 25738
aimé: 24842
fruit: 23626
préféré: 22886
agréable: 17751
froid: 16794
son: 16496
chaud: 16405
de: 15070
plus: 14934
automne: 14727
mois: 14350
à: 13870
elle: 12056
citrons: 11679
paris: 11334
inde: 11277
états-unis: 11210
france: 11170
jersey: 11052
new: 11047
chine: 10936
pendant: 10741
pamplemousse: 10140
mon: 9403
votre: 9368
juin: 9133
printemps: 9100
janvier: 9090
hiver: 9038
mars: 9023
été: 8999
mai: 8995
septembre: 8958
juillet: 8956
avril: 8954
novembre: 8951
décembre: 8945
février: 8942
octobre: 8911
aime: 8870
août: 8789
merveilleux: 8704
relaxant: 8458
doux: 8458
humide: 8446
notre: 8319
californie: 8189
sec: 7957
leur: 7855
occupé: 7782
pluvieux: 7658
calme: 7256
beau: 6387
habituellement: 6215
pommes: 5844
pêches: 5844
oranges: 5844
poires: 5844
fraises: 5844
bananes: 5844
verts: 5835
rais

In [28]:
df_fr_en["text_en"]

0         new jersey is sometimes quiet during autumn , ...
1         the united states is usually chilly during jul...
2         california is usually quiet during march , and...
3         the united states is sometimes mild during jun...
4         your least liked fruit is the grape , but my l...
                                ...                        
137855    france is never busy during march , and it is ...
137856    india is sometimes beautiful during spring , a...
137857    india is never wet during summer , but it is s...
137858    france is never chilly during january , but it...
137859    the orange is her favorite fruit , but the ban...
Name: text_en, Length: 137860, dtype: object

## RNN

In [31]:
from RNN_translator import RNNTranslator, grid_search_rnn

# Define parameter grid
param_grid = {
    'embedding_dim': [64, 128],
    'hidden_units': [512],
    'dropout_rate': [0.1],
    'learning_rate': [0.001, 0.0001]
}

# Run grid search
best_configs = grid_search_rnn(df_fr_en, param_grid, epochs=30, patience=5, n_best=3)
# Train with best config
best_params = best_configs[0]['params']
rnn = RNNTranslator(**best_params)

Starting RNN Grid Search...
Parameter grid: {'embedding_dim': [64, 128], 'hidden_units': [512], 'dropout_rate': [0.1], 'learning_rate': [0.001, 0.0001]}
Total configurations to test: 4

Configuration 1/4
Parameters: {'dropout_rate': 0.1, 'embedding_dim': 64, 'hidden_units': 512, 'learning_rate': 0.001}
Processing tokenized data...
Dataset size after filtering: 137860
Training set: 109288
Test set: 27572
Validation set: 1000
Building vocabularies...
French vocabulary size: 361
English vocabulary size: 204
Converting tokens to sequences...
Building simple RNN model...
Model built with 731,660 parameters
Starting training for maximum 30 epochs with patience 5...
Epoch 1/30
214/214 ━━━━━━━━━━━━━━━━━━━━ 261s 1s/step - accuracy: 0.3862 - loss: 1.1462 - val_accuracy: 0.4049 - val_loss: 0.7251
Epoch 2/30
214/214 ━━━━━━━━━━━━━━━━━━━━ 256s 1s/step - accuracy: 0.3943 - loss: 0.7066 - val_accuracy: 0.4191 - val_loss: 0.6356
Epoch 3/30
214/214 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.4069 - 

configuration 1 completed:
   Test Accuracy: 0.3525
   Test Loss: 1.9067
   Total Parameters: 731,660


## Faster LSTM ATTEMPT

In [39]:
import os, tensorflow as tf

# Threading: use all cores (tune if oversubscribed)
tf.config.threading.set_intra_op_parallelism_threads(0)  # 0 lets TF choose
tf.config.threading.set_inter_op_parallelism_threads(0)

# Enable XLA JIT for graphs (big win for RNNs/LSTMs/Transformers)
tf.config.optimizer.set_jit(True)  # or tf.config.experimental.enable_mlir_graph_optimization()

# Optional: ensure deterministic (can hurt perf)
# os.environ["TF_DETERMINISTIC_OPS"] = "0"

# If your train_translator_from_tokens accepts datasets, use tf.data optimizations.
# Otherwise, just leave it — XLA+threads still helps.
tf.random.set_seed(42)
with tf.device('/CPU:0'):
    translator, history = train_translator_from_tokens(
        df_fr_en,
        validation_size=1000,
        # if you expose these inside the function, use:
        # batch_size=..., shuffle_buffer=..., num_parallel_calls=tf.data.AUTOTUNE, prefetch=tf.data.AUTOTUNE,
    )

translator.save_model('lstm_translator_cpu_xla')

RuntimeError: Intra op parallelism cannot be modified after initialization.